In [3]:
import tkinter as tk

In [4]:
def close_app():
    window.destroy()

# Create the main window
window = tk.Tk()
window.title("Tkinter Example")

# Create a button and add it to the window
close_button = tk.Button(window, text="Close", command=close_app)
close_button.pack(padx=20, pady=20)

# Start the main event loop
window.mainloop()

In [ ]:
pip install pysolr

In [1]:
import csv
import pysolr

In [2]:
solr = pysolr.Solr('http://localhost:8983/solr/631Project', timeout=10)

In [3]:
# Define the query parameters
params = {
    'q': 'Title:Harry Potter',  # Query string
    'rows': 10,  # Number of results to return
    'fl': 'Title,Author,Total_Vote',  # Fields to return
}

# Execute the query and get the results
results = solr.search(**params)

# Print the results
for result in results:
    print(f"Title: {result['Title']}, Author: {result['Author']}, Total_Vote: {result['Total_Vote']}")

Title: [' Harry Ponderous\n'], Author: ['Lejink'], Total_Vote: [2]
Title: [' RIP Harry!\n'], Author: ['uds3'], Total_Vote: [208]
Title: [' Awesome Harry Potter\n'], Author: ['Students_SimoneVeil'], Total_Vote: [3]
Title: [' Agree with Harry\n'], Author: ['ladylegend_723'], Total_Vote: [17]
Title: [' Hi ya Harry!\n'], Author: ['itfrede'], Total_Vote: [0]
Title: [' DIRTY HARRY MOVE OVER\n'], Author: ['nogodnomasters'], Total_Vote: [5]
Title: [' starring Harry Potter... huh?\n'], Author: ['Quinoa1984'], Total_Vote: [5]
Title: [' Harry Potter meets the Incredibles.\n'], Author: ['delsol-1'], Total_Vote: [104]
Title: [' Parisian dirty harry done poorly\n'], Author: ['filmalamosa'], Total_Vote: [15]
Title: [' Felt Like Harry Potter Again\n'], Author: ['Rainey-Dawn'], Total_Vote: [2]


In [2]:
import pandas as pd

ratings = pd.read_csv('C:/Users/Ziheng/Desktop/COMP 631/631 project/ml-25m/ratings.csv')
movies = pd.read_csv('C:/Users/Ziheng/Desktop/COMP 631/631 project/ml-25m/movies.csv')

In [8]:
movies['release_year'] = movies['title'].str.extract(r'(?:\((\d{4})\))?\s*$', expand=False)

In [9]:
movies = movies.rename(columns={'title': 'movie_names'})

In [10]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [11]:
movies.head()

,movieId,movie_names,genres,release_year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II (1995),Comedy,1995


In [ ]:
pip install surprise

In [3]:
from surprise import Reader, Dataset, SVD, SVDpp
from surprise import accuracy

In [4]:
reader = Reader(rating_scale=(1, 5))

dataset = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader=reader)

svd = SVD(n_factors=50)

In [5]:
trainset = dataset.build_full_trainset()

svd.fit(trainset)

In [12]:
id_2_names = dict()

for idx, names in zip(movies['movieId'], movies['movie_names']):
    id_2_names[idx] = names

In [13]:
def Build_Anti_Testset4User(user_id):
    
    fill = trainset.global_mean
    anti_testset = list()
    u = trainset.to_inner_uid(user_id)
    
    # ur == users ratings
    user_items = set([item_inner_id for (item_inner_id, rating) in trainset.ur[u]])
    
    anti_testset += [(trainset.to_raw_uid(u), trainset.to_raw_iid(i), fill) for
                            i in trainset.all_items() if i not in user_items]
    
    return anti_testset

In [14]:
def TopNRecs_SVD(user_id, num_recommender=10, latest=False):
    
    testSet = Build_Anti_Testset4User(user_id)
    predict = svd.test(testSet)
    
    recommendation = list()
    
    for userID, movieID, actualRating, estimatedRating, _ in predict:
        intMovieID = int(movieID)
        recommendation.append((intMovieID, estimatedRating))
        
    recommendation.sort(key=lambda x: x[1], reverse=True)
    
    movie_names = []
    movie_ratings = []
    
    for name, ratings in recommendation[:20]:
        movie_names.append(id_2_names[name])
        movie_ratings.append(ratings)
        
    movie_dataframe =  pd.DataFrame({'movie_names': movie_names,
                                     'rating': movie_ratings}).merge(movies[['movie_names', 'release_year']],
                                            on='movie_names', how='left')
    
    #recall = tf.compat.v1.metrics.recall_at_k(predictions=predict, k=num_recommender)
    #precision = accuracy.precision_at_k(predict, k=num_recommender, threshold=3.5)
    #roc_auc = accuracy.roc_auc_score(predict)
    #hit = accuracy.hit_rate(predict, k=num_recommender, threshold=3.5)
    #ndcg = accuracy.ndcg_at_k(predict, k=num_recommender, threshold=3.5)
    
    #print(f'Recall@{k}: {recall:.4f}')
    #print(f'Precision@{k}: {precision:.4f}')
    #print(f'ROC AUC: {roc_auc:.4f}')
    #print(f'Hit Rate@{k}: {hit:.4f}')
    #print(f'nDCG@{k}: {ndcg:.4f}')
    
    
    if latest == True:
        return movie_dataframe.sort_values('release_year', ascending=False)[['movie_names', 'rating']].head(num_recommender)
    
    else:
        return movie_dataframe.drop('release_year', axis=1).head(num_recommender)

In [17]:
from surprise import dump

In [18]:
# Save the model to a file
file_name = '631Project_svd_model.joblib'
dump.dump(file_name, algo=svd)

In [19]:
# Load the saved model
file_name = '631Project_svd_model.joblib'
loaded_algo, loaded_trainset = dump.load(file_name)

In [24]:
def TopNRecs_SVD(user_id, num_recommender=10, latest=False):
    
    testSet = Build_Anti_Testset4User(user_id)
    predict = loaded_algo.test(testSet)
    
    recommendation = list()
    
    for userID, movieID, actualRating, estimatedRating, _ in predict:
        intMovieID = int(movieID)
        recommendation.append((intMovieID, estimatedRating))
        
    recommendation.sort(key=lambda x: x[1], reverse=True)
    
    movie_names = []
    movie_ratings = []
    
    for name, ratings in recommendation[:20]:
        movie_names.append(id_2_names[name])
        movie_ratings.append(ratings)
        
    movie_dataframe =  pd.DataFrame({'movie_names': movie_names,
                                     'rating': movie_ratings}).merge(movies[['movie_names', 'release_year']],
                                            on='movie_names', how='left')
    
    #recall = tf.compat.v1.metrics.recall_at_k(predictions=predict, k=num_recommender)
    #precision = accuracy.precision_at_k(predict, k=num_recommender, threshold=3.5)
    #roc_auc = accuracy.roc_auc_score(predict)
    #hit = accuracy.hit_rate(predict, k=num_recommender, threshold=3.5)
    #ndcg = accuracy.ndcg_at_k(predict, k=num_recommender, threshold=3.5)
    
    #print(f'Recall@{k}: {recall:.4f}')
    #print(f'Precision@{k}: {precision:.4f}')
    #print(f'ROC AUC: {roc_auc:.4f}')
    #print(f'Hit Rate@{k}: {hit:.4f}')
    #print(f'nDCG@{k}: {ndcg:.4f}')
    
    
    if latest == True:
        return movie_dataframe.sort_values('release_year', ascending=False)[['movie_names', 'rating']].head(num_recommender)
    
    else:
        return movie_dataframe.drop('release_year', axis=1).head(num_recommender)

In [ ]:
TopNRecs_SVD(20000, num_recommender=10)